# Rotation Curves: End-to-End Halo Construction and Eight Experiments

This notebook contains halo construction, tracer initialization, orbit integration,
diagnostics, and MP4 rendering. It does not import the separate rotation runner,
freezing, or rendering scripts. Halo construction uses the tracked `fdmtools.py`
module and the installed scientific libraries.

**Run in this order:** shared definitions -> experiment parameters -> prepare/build
halo -> evolve tracers -> draw curves and MP4. Configuration no longer requires
pre-existing halo files. With `build_halo = True`, a missing halo is constructed in
exactly the directory subsequently read by both evolving and frozen experiments.
Complete existing halos are reused without reconstruction. `build_halo = False`
requires a complete existing history. Plotting cells only read complete trajectories;
they do not start a new halo construction or particle integration.

Each run selects one experiment: four evolving cases E1-E4 and four frozen cases
F1-F4. Defaults are 600,000 tracers, seed 0, 0.1 Gyr outputs, and an 11 Gyr endpoint.
Every case provides an MP4 and overlaid profiles at 0, 1, 5, and 11 Gyr.
Changing `f` changes the fractional Gaussian scatter of the initial tangential
speed magnitude; the initial radial velocity is zero.

New integrations use identical seeded base positions and tangential directions
across scatter cases and keep halo-centered inertial coordinates. Archived evolving
trajectories can be reused read-only, but have tracer-centered stored positions and,
for legacy no-scatter data, nominal times. Set `reuse_existing = False` to bypass
those archives and produce matched new trajectories. Complete notebook chunks can
still resume.

For each selected mass, construction and evolution use the same
`ma...M...rres.../s0/` directory, with no extra `_curve` suffix. Rotation products
are saved below `rotation_notebook/evolving/` or `rotation_notebook/freeze1.0Gyr/`.
Filenames include particle count, scatter, seed, and the selected freeze time.

Newly built histories include the final 11 Gyr halo snapshot. Existing histories
that end at 10.9 Gyr retain their original last-snapshot hold to 11 Gyr.
The density ramp remains `(1-s)*smooth_baseline + s*FDM_density`, with
`s = min(t / 1 Gyr, 1)` by default. Freezing holds the full nonspherical potential
of the chosen snapshot, not a spherical average. Freeze time must match both a
halo snapshot and an orbit output; it is never silently rounded.

# 0. Run Shared Definitions First: No Construction or Orbit Integration

### Runtime and Experiment Configuration

In [ ]:
import ctypes
import hashlib
import json
import os
import time
from pathlib import Path

os.environ.setdefault('OMP_NUM_THREADS', '12')
import numpy as np
for library in ('libgslcblas.so.0', 'libgsl.so.27'):
    candidate = Path.home()/'.local/opt/gsl/lib'/library
    if candidate.exists():
        ctypes.CDLL(str(candidate), mode=ctypes.RTLD_GLOBAL)
import agama as ag
ag.setUnits(mass=1, length=1, velocity=1)

HALO_MODELS = {
    1e9: 'ma1.0e-22M1.0e+09ts0.1rres512',
    1e10: 'ma1.0e-22M1.0e+10ts0.1rres256',
}
TIME_UNIT_GYR = 0.978  # Same conversion used by the archived halo timestamp tables.


def number_tag(value, minimum_decimals=1):
    text = np.format_float_positional(float(value), trim='-')
    if '.' not in text:
        text += '.'
    return text + '0'*max(0, minimum_decimals-len(text.split('.')[1]))


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)


def make_rotation_config(halo_mass=1e10, f=0.1, freeze_time_gyr=None,
                         nparticle=600000, seed=0, end_time_gyr=11.,
                         output_dt_gyr=.1, chunk_size=100000, accuracy=1e-8,
                         root=None, halo_folder=None, reuse_existing=True,
                         output_dir=None, min_count=40, build_if_missing=True,
                         halo_build_settings=None):
    root = Path(root or Path.cwd()).resolve()
    mass = float(halo_mass)
    if mass not in HALO_MODELS:
        raise ValueError('Supported halo masses are 1e9 and 1e10 solar masses.')
    if halo_folder is not None:
        known_mass = next((m for m,p in HALO_MODELS.items() if Path(halo_folder).name==p),None)
        if known_mass is not None and known_mass != mass:
            raise ValueError('halo_mass and halo_folder select different archived halos.')
    if not np.isfinite(f) or f < 0:
        raise ValueError('f must be a finite, non-negative fractional speed scatter.')
    if int(nparticle) != nparticle or nparticle <= 0 or int(chunk_size) != chunk_size or chunk_size <= 0:
        raise ValueError('Particle and chunk counts must be positive integers.')
    if not np.isfinite(accuracy) or accuracy <= 0:
        raise ValueError('accuracy must be finite and positive.')
    if not np.isfinite(output_dt_gyr) or output_dt_gyr <= 0 or not np.isfinite(end_time_gyr) or end_time_gyr <= 0:
        raise ValueError('Output cadence and end time must be finite and positive.')
    steps = int(round(end_time_gyr/output_dt_gyr))
    if not np.isclose(steps*output_dt_gyr, end_time_gyr, rtol=0, atol=1e-10):
        raise ValueError('End time must be an integer multiple of the output cadence.')
    if freeze_time_gyr is not None:
        freeze_time_gyr = float(freeze_time_gyr)
        if not np.isfinite(freeze_time_gyr) or not 0 <= freeze_time_gyr < end_time_gyr:
            raise ValueError('Freeze time must satisfy 0 <= freeze time < end time.')
        if not np.isclose(round(freeze_time_gyr/output_dt_gyr)*output_dt_gyr,
                          freeze_time_gyr, rtol=0, atol=1e-10):
            raise ValueError('Freeze time must coincide with an orbit output time.')
    halo = root/Path(halo_folder or HALO_MODELS[mass])/f's{int(seed)}'
    history = 'evolving' if freeze_time_gyr is None else f'freeze{number_tag(freeze_time_gyr)}Gyr'
    tag = f'Np{int(nparticle)}_vscatter{number_tag(f,3)}'
    if freeze_time_gyr is not None:
        tag += f'_freeze{number_tag(freeze_time_gyr)}Gyr'
    tag += f'_s{int(seed)}'
    output = Path(output_dir) if output_dir is not None else halo/'rotation_notebook'/history
    params = dict(halo_mass=mass, f=float(f), freeze_time_gyr=freeze_time_gyr,
                  nparticle=int(nparticle), seed=int(seed), end_time_gyr=float(end_time_gyr),
                  output_dt_gyr=float(output_dt_gyr), chunk_size=int(chunk_size),
                  accuracy=float(accuracy), min_count=int(min_count),
                  coordinate_frame='Unshifted halo-centered inertial frame; no recentering',
                  implementation='rotation-notebook-v1')
    return dict(params, root=root, halo=halo, output=output, tag=tag,
                time_gyr=np.arange(steps+1)*output_dt_gyr,
                radius_edges=np.linspace(0,30,91), reuse_existing=bool(reuse_existing),
                build_if_missing=bool(build_if_missing), halo_build_settings=dict(halo_build_settings or {}))

### Halo Construction, Potential Histories, and Initial Conditions

In [ ]:
def read_potential_table(config):
    table = []
    for line in (config['halo']/'pot_evolving_gradual.ini').read_text().splitlines():
        fields = line.split()
        if len(fields) != 2:
            continue
        try:
            timestamp = float(fields[0])
        except ValueError:
            continue
        table.append((timestamp, config['halo']/fields[1]))
    if not table or np.any(np.diff([row[0] for row in table]) <= 0):
        raise ValueError('Potential timestamps must be present and strictly increasing.')
    return table


def build_rotation_potentials(config):
    table = read_potential_table(config)
    selected = table
    fixed = None
    freeze_index = None
    if config['freeze_time_gyr'] is not None:
        epoch = config['freeze_time_gyr']
        matches = np.flatnonzero(np.isclose(np.array([r[0] for r in table])*TIME_UNIT_GYR,
                                           epoch, rtol=0, atol=1e-10))
        if not len(matches):
            raise ValueError(f'Freeze time {epoch} Gyr is not an archived halo snapshot; '
                             'the present histories have 0.1 Gyr spacing. No rounding is applied.')
        selected = table[:int(matches[0])+1]
        fixed = ag.Potential(str(selected[-1][1]))
        freeze_index = int(round(epoch/config['output_dt_gyr']))
    # Hash inputs so a changed halo cannot reuse trajectories from an older potential.
    digest = hashlib.sha256()
    for timestamp,path in selected:
        digest.update(str(timestamp).encode())
        digest.update(path.read_bytes())
    payload = {key:config[key] for key in ('halo_mass','f','freeze_time_gyr','nparticle',
        'seed','end_time_gyr','output_dt_gyr','chunk_size','accuracy','min_count','coordinate_frame','implementation')}
    payload['potential_sha256'] = digest.hexdigest()
    payload['halo_directory'] = str(config['halo'])
    manifest = config['output']/f'configuration_{config["tag"]}.json'
    if manifest.exists() and json.loads(manifest.read_text())['parameters'] != payload:
        fingerprint = hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:10]
        config['output'] = config['output']/f'run_{fingerprint}'
        manifest = config['output']/f'configuration_{config["tag"]}.json'
    config['output'].mkdir(parents=True, exist_ok=True)
    ini = config['output']/f'potential_{config["tag"]}.ini'
    ini.write_text('[Potential]\ntype=Evolving\nlinearInterp=True\nTimestamps\n'+
                   ''.join(f'{t:.17g} {path}\n' for t,path in selected))
    potential = ag.Potential(str(ini))
    points = np.array([[1.,.2,.3],[5.,2.,-1.],[10.,0.,0.],[20.,-3.,1.]])
    checks = dict(force_hold_max_abs_error=None, pre_freeze_match_max_abs_error=None)
    if fixed is not None:
        original = ag.Potential(str(config['halo']/'pot_evolving_gradual.ini'))
        hold_times = np.linspace(config['freeze_time_gyr'],config['end_time_gyr'],4)
        early_times = np.linspace(0,config['freeze_time_gyr'],4)
        hold = [np.max(np.abs(potential.force(points,t=t/TIME_UNIT_GYR)-fixed.force(points)))
                for t in hold_times]
        early = [np.max(np.abs(potential.force(points,t=t/TIME_UNIT_GYR)-
                              original.force(points,t=t/TIME_UNIT_GYR))) for t in early_times]
        if max(hold+early) > 1e-9:
            raise RuntimeError('Frozen potential failed force validation.')
        checks = dict(force_hold_max_abs_error=float(max(hold)),
                      pre_freeze_match_max_abs_error=float(max(early)))
    write_json(manifest,dict(parameters=payload,validation=checks))
    return dict(potential=potential, fixed=fixed, table=table, freeze_index=freeze_index)


def make_rotation_initial_conditions(config):
    n = config['nparticle']; seed = config['seed']
    rng = np.random.RandomState(seed)
    edges = np.linspace(0,30,601)
    counts = np.full(600,n//600); counts[:n%600] += 1
    radius = np.empty(n); start = 0
    for lo,hi,count in zip(edges[:-1],edges[1:],counts):
        radius[start:start+count] = (rng.random_sample(count)*(hi**3-lo**3)+lo**3)**(1/3)
        start += count
    rng.shuffle(radius)
    cosine = 2*rng.random_sample(n)-1
    sine = np.sqrt(np.maximum(1-cosine**2,0)); phi = 2*np.pi*rng.random_sample(n)
    position = np.column_stack([radius*sine*np.cos(phi),radius*sine*np.sin(phi),radius*cosine])
    baseline = ag.Potential(type='Multipole',density=ag.Density(str(config['halo']/'sph_harm_base_gradual.ini')))
    speed = np.sqrt(np.maximum(ag.G*np.asarray(baseline.enclosedMass(radius))/np.maximum(radius,1e-12),0))
    rhat = position/np.maximum(radius[:,None],1e-12)
    tangent = rng.normal(size=position.shape)
    tangent -= np.sum(tangent*rhat,axis=1,keepdims=True)*rhat
    tangent /= np.maximum(np.linalg.norm(tangent,axis=1)[:,None],1e-12)
    factors = np.maximum(1+np.random.default_rng(seed).normal(scale=config['f'],size=n),0)
    return np.column_stack([position,tangent*(speed*factors)[:,None]])


def halo_history_ready(config):
    """A complete baseline and every potential named in the history must exist."""
    baseline = config['halo']/'sph_harm_base_gradual.ini'
    master = config['halo']/'pot_evolving_gradual.ini'
    if not baseline.is_file() or not master.is_file():
        return False
    try:
        table = read_potential_table(config)
        return bool(table) and all(path.is_file() for _,path in table)
    except (OSError,ValueError):
        return False


def construct_rotation_halo(config, settings=None):
    """Construct a soliton + Hernquist eigenmode halo in config['halo']."""
    import healpy as hp
    import fdmtools
    import scipy.interpolate as scinterp
    from tqdm.auto import tqdm

    options = dict(r_setup_res=512 if config['halo_mass']==1e9 else 256,
        scale_radius_kpc=10., mode_radius_kpc=160., fit_radius_kpc=50.,
        ramp_time_gyr=1., outer_iterations=1, energy_bins=50,
        schwarzschild_iterations=2500, nmax=None,
        density_rmax_kpc=25., density_dr_kpc=.1,
        nside=12, base_nside=3, potential_lmax=20,
        potential_rmin_kpc=1e-4, potential_rmax_kpc=30.)
    unknown = set(settings or {})-set(options)
    if unknown:
        raise ValueError(f'Unknown halo construction settings: {sorted(unknown)}')
    options.update(settings or {})
    mass=config['halo_mass'];m_a=1e-22;seed=config['seed']
    prefix=config['halo'];prefix.mkdir(parents=True,exist_ok=True)
    if options['ramp_time_gyr']<=0:
        raise ValueError('ramp_time_gyr must be positive.')
    fdm=fdmtools.FDMTools(m_a)
    radius=np.logspace(-3,3,int(options['r_setup_res']))
    a=options['scale_radius_kpc']
    rho_envelope=mass/(2*np.pi*a**3)/(radius/a)/(1+radius/a)**3
    rc=1.6*(m_a/1e-22)**-1*(mass/1e9)**(-1/3)
    rho_core=.019e9*(m_a/1e-22)**-2*rc**-4/(1+.091*(radius/rc)**2)**8
    inner=np.flatnonzero(rho_core>rho_envelope)
    if not len(inner):
        raise ValueError('The soliton does not exceed the envelope on the target grid.')
    transition=int(inner[-1])+1
    rho_input=scinterp.CubicSpline(radius,np.r_[rho_core[:transition],rho_envelope[transition:]])
    phi,rho,total_mass=fdm.initialize_potential(rho_input,core=False,r_max=options['mode_radius_kpc'])
    fE=fdm.DF_invert(phi,rho)
    print(f'Building {mass:.0e} solar-mass halo, seed {seed}: {prefix}',flush=True)
    E,R,amplitudes,phases,_,_=fdm.solve_amps_itr(
        options['mode_radius_kpc'],options['fit_radius_kpc'],phi,rho,mass,fE,
        maxiter=int(options['outer_iterations']),method='Isotropic',
        bins=int(options['energy_bins']),schwarz_iter=int(options['schwarzschild_iterations']),
        seed=seed,plot_progress=False)
    if options['nmax'] is not None:
        n=np.arange(amplitudes.shape[0])[:,None]
        l=np.arange(amplitudes.shape[1])[None,:]
        amplitudes[n+l+1>options['nmax']]=0
    active=(E!=0)&(amplitudes!=0)
    if not np.any(active):
        raise RuntimeError('Halo amplitude reconstruction produced no active modes.')
    # Match the original 251 radii through 25 kpc at the production defaults.
    nr=int(round(options['density_rmax_kpc']/options['density_dr_kpc']))+1
    radii=np.arange(nr)*options['density_dr_kpc'];radii[0]=1e-5
    lmax=int(np.nonzero(active)[1].max())
    radial={}
    for n,l in zip(*np.nonzero(active)):
        radial[n,l]=amplitudes[n,l]*R[n,l](radii)
    baseline=np.zeros(nr)
    for (n,l),ar in radial.items():baseline+=(2*l+1)/(4*np.pi)*ar**2

    def export_density(path,maps,nside):
        # Retain the existing HEALPix -> AGAMA real-harmonic normalization.
        harmonic_lmax=3*int(nside)-1
        rho_lm=np.array([hp.map2alm(row,lmax=harmonic_lmax,iter=3) for row in maps])
        with path.open('w') as stream:
            stream.write('[Density]\ntype=DensitySphericalHarmonic\n'
                         f'gridSizeR={nr}\nlmax={harmonic_lmax}\nCoefficients\n#rho\n#radius')
            for l in range(harmonic_lmax+1):
                for m in range(-l,l+1):stream.write(f' l={l},m={m}')
            stream.write('\n')
            for i,r in enumerate(radii):
                stream.write(f'{r:.10g}')
                for l in range(harmonic_lmax+1):
                    for m in range(-l,l+1):
                        z=rho_lm[i,hp.Alm.getidx(harmonic_lmax,l,abs(m))]
                        value=(z.real if m==0 else np.sqrt(2)*z.real if m>0 else -np.sqrt(2)*z.imag)/np.sqrt(4*np.pi)
                        stream.write(f' {value:.10g}')
                stream.write('\n')

    base_maps=np.broadcast_to(baseline[:,None],(nr,hp.nside2npix(options['base_nside'])))
    export_density(prefix/'sph_harm_base_gradual.ini',base_maps,options['base_nside'])
    rows=[]
    for i,t in enumerate(tqdm(config['time_gyr'],desc='Halo snapshots')):
        real=np.zeros((nr,hp.Alm.getsize(lmax)),dtype=complex)
        imag=np.zeros_like(real)
        for (n,l),ar in radial.items():
            phase=-E[n,l]/fdm.h_over_m*1.022712165045695*t
            for m in range(l+1):
                index=hp.Alm.getidx(lmax,l,m)
                plus=np.exp(1j*(phases[n,l,m]+phase))
                minus=np.exp(-1j*(phases[n,l,-m]+phase))*(-1.)**m
                if m==0:
                    real[:,index]+=ar*plus
                    imag[:,index]+=ar*plus/1j
                else:
                    real[:,index]+=ar*(plus+minus)/2
                    imag[:,index]+=ar*(plus-minus)/(2j)
        weight=min(float(t)/options['ramp_time_gyr'],1.)
        maps=np.empty((nr,hp.nside2npix(options['nside'])))
        for j in range(nr):
            wave=hp.alm2map(real[j],options['nside'])+1j*hp.alm2map(imag[j],options['nside'])
            maps[j]=weight*np.abs(wave)**2
        density_path=prefix/f'sph_harm_gradual{i:03d}.ini'
        export_density(density_path,maps,options['nside'])
        density=ag.Density(str(density_path))
        if weight<1:
            base_path=prefix/f'sph_harm_base_gradual{i:03d}.ini'
            export_density(base_path,base_maps*(1-weight),options['base_nside'])
            density=ag.Density(density,ag.Density(str(base_path)))
        potential=ag.Potential(type='Multipole',density=density,
            lmax=int(options['potential_lmax']),rmin=options['potential_rmin_kpc'],
            rmax=options['potential_rmax_kpc'])
        filename=prefix/f'sph_harm_pot_gradual{i:03d}.ini'
        potential.export(str(filename))
        rows.append((float(t)/TIME_UNIT_GYR,filename.name))
    # Publish the master table only after all its snapshots have been exported.
    master=prefix/'pot_evolving_gradual.ini'
    temporary=master.with_suffix('.ini.tmp')
    temporary.write_text('[Potential]\ntype=Evolving\nlinearInterp=True\nTimestamps\n'+
                         ''.join(f'{t:.17g} {name}\n' for t,name in rows))
    temporary.replace(master)
    write_json(prefix/'halo_build_complete.json',dict(halo_mass_label=mass,fdm_mass_ev=m_a,
        seed=seed,soliton_radius_kpc=rc,initial_target_mass=total_mass,settings=options,
        time_gyr=config['time_gyr'].tolist(),
        density_ramp='(1-s)*smooth modal baseline + s*FDM density; s=min(t/ramp_time,1)'))
    print(f'Halo history ready: {master}',flush=True)
    return prefix


def prepare_rotation_halo(config,build_if_missing=True,settings=None):
    """Use one directory for construction, evolving orbits, and frozen controls."""
    if halo_history_ready(config):
        table=read_potential_table(config)
        last=table[-1][0]*TIME_UNIT_GYR
        # Legacy archives omit the endpoint at 11 Gyr, but hold their final snapshot.
        spacing=max(np.diff([t*TIME_UNIT_GYR for t,_ in table]),default=config['output_dt_gyr'])
        if last+spacing+1e-9 < config['end_time_gyr']:
            raise ValueError('The existing halo history is too short for this experiment.')
        print(f'Using complete halo history: {config["halo"]}',flush=True)
        return config['halo']
    if not build_if_missing:
        raise FileNotFoundError(f'Missing or incomplete halo history in {config["halo"]}. '
                                'Enable build_halo to construct it before orbit integration.')
    return construct_rotation_halo(config,settings=settings)

### Trajectory Reuse and Particle Evolution

In [ ]:
def validate_orbit_files(files,config,require_markers=False):
    if not files:
        return False
    total = 0
    for path in files:
        if require_markers and not path.with_suffix('.complete.json').exists():
            return False
        try:
            a = np.load(path,mmap_mode='r')
            if a.ndim != 3 or a.shape[:2] != (len(config['time_gyr']),6):
                return False
            if not np.all(np.isfinite(a[[0,-1]])) or np.any(np.all(a[-1]==0,axis=0)):
                return False
            total += a.shape[2]
        except (ValueError,OSError):
            return False
    return total == config['nparticle']


def existing_rotation_files(config):
    managed = sorted(config['output'].glob(f'xvta_{config["tag"]}_part*.npy'))
    if validate_orbit_files(managed,config,require_markers=True):
        return managed, 'Verified notebook trajectories; halo-centered coordinates and explicit times.'
    if not config['reuse_existing']:
        return None,None
    if config['freeze_time_gyr'] is None:
        tag = f'Np{config["nparticle"]}_vscatter{number_tag(config["f"],3)}'
        choices = [sorted(config['halo'].glob(f'xvta_{tag}_part*.npy'))]
        if config['f']==0:
            choices.append(sorted(config['halo'].glob(f'xvta_Np{config["nparticle"]}_part*.npy')))
        # Old runs only represent the documented seed-0, 11 Gyr, 0.1 Gyr setup.
        if config['seed']!=0 or config['end_time_gyr']!=11 or config['output_dt_gyr']!=.1 or config['accuracy']!=1e-8:
            return None,None
        for files in choices:
            if validate_orbit_files(files,config):
                return files, ('Legacy evolving archive, reused read-only: positions were tracer-centered; '
                               'velocities were not shifted. Legacy no-scatter times are nominal.')
    else:
        folder = config['halo']/f'freeze{number_tag(config["freeze_time_gyr"])}Gyr'
        old_config = folder/'configuration.json'
        if old_config.exists():
            old = json.loads(old_config.read_text())
            expected = {'halo_mass_label':config['halo_mass'],'nparticle':config['nparticle'],
                        'seed':config['seed'],'freeze_time_gyr':config['freeze_time_gyr'],
                        'end_time_gyr':config['end_time_gyr'],'output_dt_gyr':config['output_dt_gyr'],
                        'integration_accuracy':config['accuracy']}
            files = sorted(folder.glob(f'xvta_{config["tag"]}_part*.npy'))
            if (all(old.get(k)==v for k,v in expected.items()) and
                    config['f'] in old.get('scatter_fractions',[]) and
                    validate_orbit_files(files,config,require_markers=True)):
                return files, 'Verified frozen-control archive; halo-centered coordinates and explicit times.'
    return None,None


def evolve_rotation(config,render_only=False):
    prepare_rotation_halo(config,
        build_if_missing=config['build_if_missing'] and not render_only,
        settings=config['halo_build_settings'])
    state = build_rotation_potentials(config)
    files, provenance = existing_rotation_files(config)
    if files is None:
        if render_only:
            raise FileNotFoundError('No complete compatible trajectory set; run the evolution cell first.')
        ic = make_rotation_initial_conditions(config); files=[]
        for k,start in enumerate(range(0,config['nparticle'],config['chunk_size'])):
            subset=ic[start:start+config['chunk_size']]
            path=config['output']/f'xvta_{config["tag"]}_part{k:03d}.npy'
            done=path.with_suffix('.complete.json')
            small=dict(config,nparticle=len(subset))
            if not validate_orbit_files([path],small,require_markers=True):
                print(f'Integrating chunk {k+1}: {len(subset):,} particles',flush=True)
                begin=time.perf_counter()
                result=ag.orbit(potential=state['potential'],ic=subset,
                    time=config['end_time_gyr']/TIME_UNIT_GYR,timestart=0.,
                    trajsize=len(config['time_gyr']),accuracy=config['accuracy'],dtype='double')
                if not np.allclose(result[0,0]*TIME_UNIT_GYR,config['time_gyr'],rtol=0,atol=1e-10):
                    raise RuntimeError('AGAMA returned unexpected trajectory times.')
                a=np.lib.format.open_memmap(path,mode='w+',dtype='float32',
                                           shape=(len(config['time_gyr']),6,len(subset)))
                for j,orbit in enumerate(result):
                    a[:,:,j]=orbit[1]
                a.flush();del a,result
                write_json(done,dict(nparticle=len(subset),scatter=config['f'],
                                    elapsed_seconds=time.perf_counter()-begin))
            files.append(path)
            write_json(config['output']/f'status_{config["tag"]}.json',dict(stage='integrating',completed_chunks=k+1))
        provenance='New notebook trajectories; halo-centered coordinates and explicit times.'
    print(f'{config["halo_mass"]:.0e} solar masses, f={config["f"]}, freeze={config["freeze_time_gyr"]}: {provenance}')
    return dict(config=config,state=state,files=files,provenance=provenance)

### Full-Population Diagnostics

In [ ]:
def rotation_diagnostics(case):
    cfg=case['config']; state=case['state']
    path=cfg['output']/f'diagnostics_{cfg["tag"]}.npz'
    source_fingerprint = hashlib.sha256(json.dumps([
        (str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in case['files']
    ]).encode()).hexdigest()
    if path.exists():
        cached = dict(np.load(path))
        if str(cached.get('source_fingerprint','')) == source_fingerprint:
            case['diagnostics']=cached;return cached
    maps=[np.load(p,mmap_mode='r') for p in case['files']]
    nt=len(cfg['time_gyr']); edges=cfg['radius_edges']; r=.5*(edges[:-1]+edges[1:])
    mean=np.full((nt,len(r)),np.nan); std=mean.copy(); phi=mean.copy(); vc=mean.copy()
    v2s=[]; forces=[]; rvs=[]; bv2=[]; bf=[]; brv=[]
    bound=[]; e0=[]; fi=state['freeze_index']; cache={}
    def energy(xv):
        return .5*np.sum(xv[:,3:]**2,axis=1)+state['fixed'].potential(xv[:,:3])
    if fi is not None:
        for a in maps:
            energies=energy(a[fi].T.astype(float));e0.append(energies);bound.append(energies<0)
    times=np.array([q[0]*TIME_UNIT_GYR for q in state['table']])
    def reference_mass(t):
        t=min(t,cfg['freeze_time_gyr']) if fi is not None else t
        hi=int(np.searchsorted(times,t,side='right'));hi=min(max(hi,1),len(times)-1)
        lo=hi-1; weight=float(np.clip((t-times[lo])/(times[hi]-times[lo]),0,1))
        for j in (lo,hi):
            if j not in cache:
                cache[j]=np.asarray(ag.Potential(str(state['table'][j][1])).enclosedMass(r))
        return (1-weight)*cache[lo]+weight*cache[hi]
    for ti,t in enumerate(cfg['time_gyr']):
        count=np.zeros(len(r)); total=count.copy(); total2=count.copy(); phitotal=count.copy()
        all_v2=all_force=all_rv=bound_v2=bound_force=bound_rv=0.
        for k,a in enumerate(maps):
            xv=a[ti].T.astype(float);x=xv[:,:3];v=xv[:,3:]
            radii=np.linalg.norm(x,axis=1); v2=np.sum(v*v,axis=1);rv=np.sum(x*v,axis=1)
            vt=np.linalg.norm(np.cross(x,v),axis=1)/np.maximum(radii,1e-12)
            R=np.linalg.norm(x[:,:2],axis=1)
            vphi=np.abs(x[:,0]*v[:,1]-x[:,1]*v[:,0])/np.maximum(R,1e-12)
            force=(state['fixed'].force(x) if fi is not None and ti>=fi
                   else state['potential'].force(x,t=t/TIME_UNIT_GYR))
            moment=-np.sum(x*force,axis=1)
            all_v2+=v2.sum();all_force+=moment.sum();all_rv+=rv.sum()
            if fi is not None:
                b=bound[k];bound_v2+=v2[b].sum();bound_force+=moment[b].sum();bound_rv+=rv[b].sum()
            idx=np.searchsorted(edges,radii,side='right')-1
            valid=(idx>=0)&(idx<len(r));ids=idx[valid]
            count+=np.bincount(ids,minlength=len(r))
            total+=np.bincount(ids,weights=vt[valid],minlength=len(r))
            total2+=np.bincount(ids,weights=vt[valid]**2,minlength=len(r))
            # The auxiliary azimuthal curve is binned in cylindrical radius R.
            iphi=np.searchsorted(edges,R,side='right')-1
            validphi=(iphi>=0)&(iphi<len(r))
            phitotal+=np.bincount(iphi[validphi],weights=vphi[validphi],minlength=len(r))
            if k==0: phicount=np.zeros(len(r))
            phicount+=np.bincount(iphi[validphi],minlength=len(r))
        good=count>=cfg['min_count'];mean[ti,good]=total[good]/count[good]
        std[ti,good]=np.sqrt(np.maximum(total2[good]/count[good]-mean[ti,good]**2,0))
        goodphi=phicount>=cfg['min_count'];phi[ti,goodphi]=phitotal[goodphi]/phicount[goodphi]
        vc[ti]=np.sqrt(np.maximum(ag.G*reference_mass(t)/r,0))
        v2s.append(all_v2/cfg['nparticle']);forces.append(all_force/cfg['nparticle']);rvs.append(all_rv/cfg['nparticle'])
        if fi is not None:
            nb=sum(b.sum() for b in bound)
            bv2.append(bound_v2/nb if nb else np.nan);bf.append(bound_force/nb if nb else np.nan);brv.append(bound_rv/nb if nb else np.nan)
        if ti%10==0: print(f'Diagnostics: t={t:g} Gyr',flush=True)
    data=dict(time_gyr=cfg['time_gyr'],radius=r,mean_vt=mean,std_vt=std,mean_abs_vphi=phi,vc=vc,
              source_fingerprint=source_fingerprint,
              mean_v2=np.array(v2s),mean_minus_r_dot_force=np.array(forces),mean_r_dot_v=np.array(rvs))
    if fi is not None:
        e0=np.concatenate(e0);ef=np.concatenate([energy(a[-1].T.astype(float)) for a in maps])
        error=np.abs(ef-e0)/np.maximum(np.abs(e0),1e-4*np.median(np.abs(e0)))
        data.update(bound_mean_v2=np.array(bv2),bound_mean_minus_r_dot_force=np.array(bf),
                    bound_mean_r_dot_v=np.array(brv),bound_fraction=sum(b.sum() for b in bound)/cfg['nparticle'],
                    energy_error_quantiles=np.quantile(error,[.5,.99,1]))
    np.savez(path,**data);case['diagnostics']=data
    return data

### Snapshot Curves, Virial Diagnostics, and MP4 Rendering

In [ ]:
def setup_rotation_plots():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    for name in ('times.ttf','timesbd.ttf','timesi.ttf','timesbi.ttf'):
        path=Path('/mnt/c/Windows/Fonts')/name
        if path.exists():font_manager.fontManager.addfont(str(path))
    plt.rcParams.update({'font.family':'Times New Roman','font.size':11,
                         'mathtext.fontset':'stix','axes.grid':True,'grid.alpha':.15})
    return plt


def plot_rotation_snapshots(case, snapshot_times=(0.,1.,5.,11.)):
    plt=setup_rotation_plots();data=case.get('diagnostics') or rotation_diagnostics(case);cfg=case['config']
    fig,(ax,ratio)=plt.subplots(2,1,figsize=(8.5,7),sharex=True,gridspec_kw={'height_ratios':[1.4,1]})
    styles=[(':','o'),('--','s'),('-.','^'),('-',None)]
    actual=[]
    for t,(ls,marker) in zip(snapshot_times,styles):
        matches=np.flatnonzero(np.isclose(data['time_gyr'],t,atol=1e-10,rtol=0))
        if not len(matches):continue
        i=int(matches[0]);actual.append(float(t));draw=dict(color='black',ls=ls,marker=marker,
            markevery=9,ms=3.5,mfc='white',lw=1.5,label=f't = {t:g} Gyr')
        ax.plot(data['radius'],data['mean_vt'][i],**draw)
        ratio.plot(data['radius'],data['mean_vt'][i]/data['vc'][i],**draw)
        ax.plot(data['radius'],data['vc'][i],color='.6',ls=ls,lw=1,
                label=f'Circular speed, t = {t:g} Gyr')
    ax.set(ylabel='Mean tangential speed (km/s)',title=f'M = {cfg["halo_mass"]:.0e} solar masses; f = {cfg["f"]}; '+
           ('evolving halo' if cfg['freeze_time_gyr'] is None else f'halo frozen at {cfg["freeze_time_gyr"]:g} Gyr'))
    ax.legend(ncol=2,fontsize=8,frameon=False)
    ratio.axhline(1,color='.5',ls=':');ratio.set(xlim=(1,30),ylim=(.75,1.25),
        xlabel='Halo-centered spherical radius r (kpc)',ylabel=r'$\langle v_t\rangle/v_c$')
    if 'Legacy evolving' in case['provenance']:
        ratio.set_xlabel('Legacy tracer-centered spherical radius (kpc); nominal times')
    fig.tight_layout()
    stem=cfg['output']/f'rotation_snapshots_{cfg["tag"]}'
    fig.savefig(str(stem)+'.png',dpi=160);fig.savefig(str(stem)+'.pdf')
    plt.close(fig)
    return Path(str(stem)+'.png')


def summarize_rotation(case):
    data=case.get('diagnostics') or rotation_diagnostics(case);cfg=case['config'];s={}
    mask=(data['radius']>=5)&(data['radius']<=20)
    s['final_median_vt_over_vc_r5_20']=float(np.nanmedian((data['mean_vt'][-1]/data['vc'][-1])[mask]))
    s['provenance']=case['provenance']
    s['final_global_virial_ratio']=float(data['mean_v2'][-1]/data['mean_minus_r_dot_force'][-1])
    fi=case['state']['freeze_index']
    if fi is not None:
        t=data['time_gyr'][fi:]/TIME_UNIT_GYR;dt=np.diff(t)
        v2=data['bound_mean_v2'][fi:];w=data['bound_mean_minus_r_dot_force'][fi:]
        dv=np.sum(.5*(v2[1:]+v2[:-1])*dt);dw=np.sum(.5*(w[1:]+w[:-1])*dt)
        boundary=data['bound_mean_r_dot_v'][-1]-data['bound_mean_r_dot_v'][fi]
        s.update(final_bound_cumulative_virial_ratio=float(dv/dw),
                 boundary_corrected_cumulative_ratio=float((dv-boundary)/dw),
                 bound_fraction=float(data['bound_fraction']),
                 energy_relative_error_median_p99_max=data['energy_error_quantiles'].tolist())
        plt=setup_rotation_plots();fig,axes=plt.subplots(1,2,figsize=(10,3.6))
        axes[0].plot(data['time_gyr'],data['mean_v2']/data['mean_minus_r_dot_force'],color='black')
        axes[0].axvline(cfg['freeze_time_gyr'],color='.5',ls=':')
        numerator=np.cumsum(.5*(v2[1:]+v2[:-1])*dt);denominator=np.cumsum(.5*(w[1:]+w[:-1])*dt)
        axes[1].plot(data['time_gyr'][fi+1:],numerator/denominator,color='black')
        for ax in axes: ax.axhline(1,color='.5',ls=':');ax.set_xlabel('Time (Gyr)')
        axes[0].set_ylabel('Global virial ratio');axes[1].set_ylabel('Cumulative bound virial ratio')
        fig.tight_layout();fig.savefig(cfg['output']/f'virial_{cfg["tag"]}.png',dpi=150);plt.close(fig)
    if 'Legacy evolving' in case['provenance']:
        s['virial_warning']='Legacy moving-coordinate positions: do not interpret this virial ratio as an inertial-frame test.'
    s['note']='Finite-time virial balance and mean tangential speed equal to circular speed are different tests.'
    write_json(cfg['output']/f'summary_{cfg["tag"]}.json',s)
    return s


def render_rotation(case,frame_indices=None,fps=10,save_stills=True,decode=True):
    from matplotlib import animation
    import imageio_ffmpeg
    import subprocess
    plt=setup_rotation_plots();cfg=case['config'];data=case.get('diagnostics') or rotation_diagnostics(case)
    maps=[np.load(path,mmap_mode='r') for path in case['files']]
    fig,(ax,profile)=plt.subplots(2,1,figsize=(8.6,8.4),sharex=True)
    points=ax.scatter([],[],s=.5,alpha=.12,color='black',edgecolors='none',label='Displayed tracer sample')
    reference,=ax.plot(data['radius'],data['vc'][0],color='.5',ls=':',lw=2,label='Spherical circular speed')
    theory,=profile.plot(data['radius'],data['vc'][0],color='.5',ls=':',lw=2,label='Spherical circular speed')
    tangential,=profile.plot(data['radius'],data['mean_vt'][0],color='black',lw=1.6,label='Mean tangential speed')
    azimuthal,=profile.plot(data['radius'],data['mean_abs_vphi'][0],color='black',ls='--',
                             label='Mean azimuthal speed about z (binned in R)')
    band=[None]
    upper=max(float(np.nanmax(data['vc']))*1.5,float(np.nanmax(data['mean_vt']+data['std_vt']))*1.2)
    for axis in (ax,profile):axis.set(xlim=(0,30),ylim=(0,upper),ylabel='Speed (km/s)');axis.legend(fontsize=8)
    profile.set_xlabel('Spherical radius r / cylindrical radius R (kpc)')
    stride=max(1,int(np.ceil(cfg['nparticle']/80000)))
    def update(index):
        xv=np.concatenate([a[index,:,::stride].T for a in maps])[:80000].astype(float)
        x=xv[:,:3];v=xv[:,3:];r=np.linalg.norm(x,axis=1)
        vt=np.linalg.norm(np.cross(x,v),axis=1)/np.maximum(r,1e-12)
        points.set_offsets(np.column_stack([r,vt]));reference.set_ydata(data['vc'][index])
        theory.set_ydata(data['vc'][index]);tangential.set_ydata(data['mean_vt'][index])
        azimuthal.set_ydata(data['mean_abs_vphi'][index])
        if band[0] is not None:band[0].remove()
        band[0]=profile.fill_between(data['radius'],data['mean_vt'][index]-data['std_vt'][index],
                                    data['mean_vt'][index]+data['std_vt'][index],color='.7',alpha=.25)
        t=data['time_gyr'][index]
        state='evolving halo'
        if cfg['freeze_time_gyr'] is not None and t>=cfg['freeze_time_gyr']:
            state=f'halo frozen at {cfg["freeze_time_gyr"]:g} Gyr'
        ax.set_title(f'M={cfg["halo_mass"]:.0e} solar masses | f={cfg["f"]:g} | t={t:.1f} Gyr | {state}')
        if 'Legacy evolving' in case['provenance']:
            profile.set_xlabel('Legacy tracer-centered r / cylindrical R (kpc); nominal time')
        return points,reference,theory,tangential,azimuthal
    fig.tight_layout()
    if save_stills:
        for t in (0.,1.,5.,11.):
            matches=np.flatnonzero(np.isclose(data['time_gyr'],t,rtol=0,atol=1e-10))
            if len(matches):
                update(int(matches[0]));fig.savefig(cfg['output']/f'rotation_curve_{cfg["tag"]}_t{t:.1f}Gyr.png',dpi=150)
    frames=list(range(len(data['time_gyr']))) if frame_indices is None else list(frame_indices)
    if not frames or any(i<0 or i>=len(data['time_gyr']) for i in frames):
        raise ValueError('Animation frame indices are outside the trajectory.')
    destination=cfg['output']/f'vcirc_evolution_{cfg["tag"]}.mp4'
    plt.rcParams['animation.ffmpeg_path']=imageio_ffmpeg.get_ffmpeg_exe()
    movie=animation.FuncAnimation(fig,update,frames=frames,interval=1000/fps,blit=False)
    try:
        movie.save(destination,writer=animation.FFMpegWriter(fps=fps,codec='libx264',bitrate=1800,
                   extra_args=['-pix_fmt','yuv420p','-movflags','+faststart']),dpi=130)
    finally:plt.close(fig)
    if decode:
        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-v','error','-xerror','-i',str(destination),
                        '-f','null','-'],check=True,capture_output=True)
    write_json(cfg['output']/f'status_{cfg["tag"]}.json',dict(stage='complete',mp4=str(destination),
        full_decode_verified=bool(decode),frame_count=len(frames)))
    print(f'Saved MP4: {destination}',flush=True)
    return destination

# 1. Evolving Halo: Set `halo_mass` and `f` to Select Four MP4 Results

Choose the parameters in the next code cell, then run halo preparation, particle
evolution, and plotting below. The halo follows its original ramp and continues
to evolve afterward. A missing halo is constructed when `build_halo = True`.

### E1: `halo_mass = 1e9`, `f = 0.0` -> $10^9\,M_\odot$, No Scatter, Evolving-Halo MP4

### E2: `halo_mass = 1e9`, `f = 0.1` -> $10^9\,M_\odot$, 10% Scatter, Evolving-Halo MP4

### E3: `halo_mass = 1e10`, `f = 0.0` -> $10^{10}\,M_\odot$, No Scatter, Evolving-Halo MP4

### E4: `halo_mass = 1e10`, `f = 0.1` -> $10^{10}\,M_\odot$, 10% Scatter, Evolving-Halo MP4

## Evolving Parameters: Select One E1-E4 Combination Below

Mass 1e9 selects rres512; mass 1e10 selects rres256. Construction, initialization,
and plotting all use the resulting single halo directory. Each selected case
produces its own MP4 and the four time-profile curves.

In [ ]:
halo_mass = 1e10              # Solar masses: 1e9 or 1e10
f = 0.1                       # Fractional speed scatter: 0.0 or 0.1
nparticle = 600000
seed = 0
reuse_existing = True         # False: bypass legacy orbit archives; notebook chunks can still resume
build_halo = True             # Construct a missing halo; a complete existing halo is reused
halo_build_settings = {}       # Empty: use the original production construction settings

evolving_config = make_rotation_config(
    halo_mass=halo_mass, f=f, nparticle=nparticle, seed=seed,
    reuse_existing=reuse_existing, build_if_missing=build_halo,
    halo_build_settings=halo_build_settings,
)
print('Halo construction and orbit input:', evolving_config['halo'])
print('Rotation outputs:', evolving_config['output'])

## Prepare the Selected Halo: `build_halo = True` Builds Missing Files

This cell uses a complete existing halo or constructs a missing one. The builder
creates the soliton + Hernquist target, solves radial modes and amplitudes with
`fdmtools`, reconstructs the time-dependent density with HEALPix, applies the
0-1 Gyr density ramp, exports AGAMA multipole snapshots, and finally writes their
evolving-potential table. These are the exact files read by the subsequent cells.
The potential export loop runs directly in the notebook and does not require
multiprocessing workers to pickle notebook functions.

`halo_build_settings = {}` preserves the original production numerical settings:
160 kpc mode radius, 50 kpc fitting radius, one outer reconstruction iteration,
50 energy bins, 2,500 Schwarzschild iterations, density radii through 25 kpc,
HEALPix nside 12, and potential lmax 20 through 30 kpc. Small numerical settings
used for integration smoke tests are not substitutes for these production values.

In [ ]:
prepare_rotation_halo(
    evolving_config, build_if_missing=build_halo, settings=halo_build_settings,
)

## Evolve Selected E1-E4 Tracers to 11 Gyr

Reuse complete compatible trajectories or integrate restartable particle chunks.
The evolution function also checks the same halo directory before integration.

In [ ]:
evolving_case = evolve_rotation(evolving_config)

## Render Selected E1-E4: MP4 and Profiles at 0, 1, 5, and 11 Gyr

Rerun the parameter cell after changing mass or scatter. With complete trajectories,
this plotting cell can be used directly; otherwise run halo preparation and evolution
first. Profile statistics use all particles, while the movie displays at most 80,000
individual tracers. MP4 encoding uses H.264, yuv420p, and faststart, followed by a
complete decode check. The auxiliary azimuthal speed is about z,
$|v_\phi|=|L_z|/R$, binned in cylindrical radius R; it is not a line-of-sight velocity.

In [ ]:
evolving_case = evolve_rotation(evolving_config, render_only=True)
rotation_diagnostics(evolving_case)
evolving_profiles = plot_rotation_snapshots(evolving_case, snapshot_times=(0, 1, 5, 11))
evolving_summary = summarize_rotation(evolving_case)
evolving_mp4 = render_rotation(evolving_case)
from IPython.display import display, Image, Video
display(Image(filename=str(evolving_profiles)))
display(Video(filename=str(evolving_mp4), embed=True))
print(json.dumps(evolving_summary, indent=2))
print('MP4:', evolving_mp4)

# 2. Frozen Halo: Set `halo_mass`, `f`, and `freeze_time_gyr` (Default 1.0 Gyr)

Run the shared definitions first; the evolving experiment is not required.
The frozen chapter can also start without halo files: `build_halo = True` prepares
the full history in its own selected halo directory before particle integration.
F1-F4 use the original 0-1 Gyr ramp, then hold the complete 1 Gyr potential fixed
while evolving the tracers to 11 Gyr.

### F1: `halo_mass = 1e9`, `f = 0.0`, `freeze_time_gyr = 1.0` -> $10^9\,M_\odot$, No Scatter, Frozen-Halo MP4

### F2: `halo_mass = 1e9`, `f = 0.1`, `freeze_time_gyr = 1.0` -> $10^9\,M_\odot$, 10% Scatter, Frozen-Halo MP4

### F3: `halo_mass = 1e10`, `f = 0.0`, `freeze_time_gyr = 1.0` -> $10^{10}\,M_\odot$, No Scatter, Frozen-Halo MP4

### F4: `halo_mass = 1e10`, `f = 0.1`, `freeze_time_gyr = 1.0` -> $10^{10}\,M_\odot$, 10% Scatter, Frozen-Halo MP4

## Frozen Parameters: Select One F1-F4 Combination Below

Use `freeze_time_gyr = 2.0`, for example, to hold the 2 Gyr snapshot instead.
The time must be present in both the halo history and the particle outputs
(0.1 Gyr spacing at the defaults). Each case produces one MP4 plus four curves
at 0, 1, 5, and 11 Gyr; these are four time profiles within one experiment.
Global virial balance does not require the mean tangential speed to equal the
circular speed at every radius.

In [ ]:
halo_mass = 1e10              # Solar masses: 1e9 or 1e10
f = 0.1                       # Fractional speed scatter: 0.0 or 0.1
freeze_time_gyr = 1.0          # Must coincide with a halo snapshot and orbit output
nparticle = 600000
seed = 0
reuse_existing = True
build_halo = True             # Also permits starting the frozen chapter without any halo files
halo_build_settings = {}

frozen_config = make_rotation_config(
    halo_mass=halo_mass, f=f, freeze_time_gyr=freeze_time_gyr,
    nparticle=nparticle, seed=seed, reuse_existing=reuse_existing,
    build_if_missing=build_halo, halo_build_settings=halo_build_settings,
)
print('Halo construction and orbit input:', frozen_config['halo'])
print('Rotation outputs:', frozen_config['output'])

## Prepare the Halo and Evolve Selected F1-F4: Freeze at `freeze_time_gyr`

Halo construction and frozen orbit integration use the same `frozen_config` path.
Missing files are prepared before the frozen potential is assembled.

In [ ]:
prepare_rotation_halo(
    frozen_config, build_if_missing=build_halo, settings=halo_build_settings,
)
frozen_case = evolve_rotation(frozen_config)

## Render Selected F1-F4: MP4, Four Time Profiles, and Virial Diagnostics

Energy changes are measured from the selected freeze epoch to the final time.
Force and virial diagnostics use the actual nonspherical acceleration at all radii.
The finite-time boundary-corrected virial identity is a consistency check, not an
independent proof of stationarity. The archived lower-mass controls have outer-orbit
energy-error tails; inspect the reported energy-change quantiles.

In [ ]:
frozen_case = evolve_rotation(frozen_config, render_only=True)
rotation_diagnostics(frozen_case)
frozen_profiles = plot_rotation_snapshots(frozen_case, snapshot_times=(0, 1, 5, 11))
frozen_summary = summarize_rotation(frozen_case)
frozen_mp4 = render_rotation(frozen_case)
from IPython.display import display, Image, Video
display(Image(filename=str(frozen_profiles)))
display(Video(filename=str(frozen_mp4), embed=True))
print(json.dumps(frozen_summary, indent=2))
print('MP4:', frozen_mp4)